In [ ]:
! pip uninstall -y transformers accelerate peft bitsandbytes trl

! pip install torch==2.1.2
! pip install transformers==4.38.2
! pip install accelerate==0.27.2
! pip install peft==0.10.0
! pip install trl==0.8.6

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
output_dir = "/content/drive/MyDrive/orpo_llama_model"

TinyLlama-1.1B

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"



Dataset

In [ ]:
from datasets import load_dataset
import re

MAX_TOKENS = 512

# Load dataset
dataset = load_dataset("allenai/Dolci-Think-DPO-7B", split="train")
dataset = dataset.shuffle(seed=42).select(range(50000))


# Utility functions
def truncate_text(text, max_tokens=MAX_TOKENS):
    if text is None:
        return ""
    input_ids = tokenizer(text, truncation=True, max_length=max_tokens, add_special_tokens=False)["input_ids"]
    return tokenizer.decode(input_ids, skip_special_tokens=True), len(input_ids)


def format_and_truncate(example):
    # Prompt
    prompt_trunc, prompt_len = truncate_text(example["prompt"])


    if prompt_len > MAX_TOKENS:
        return {"prompt": prompt_trunc, "chosen": "", "rejected": ""}

    # Chosen
    chosen_trunc, chosen_len = truncate_text(example["chosen"][-1]["content"])


    if chosen_len > MAX_TOKENS:
        return {"prompt": prompt_trunc, "chosen": chosen_trunc, "rejected": ""}

    # Rejected
    rejected_trunc, _ = truncate_text(example["rejected"][-1]["content"])

    return {"prompt": prompt_trunc, "chosen": chosen_trunc, "rejected": rejected_trunc}

dataset = dataset.map(format_and_truncate, remove_columns=dataset.column_names)


def is_valid(example):
    prompt_len = len(tokenizer(example["prompt"])["input_ids"])
    chosen_len = len(tokenizer(example["chosen"])["input_ids"]) if example["chosen"] else 0
    rejected_len = len(tokenizer(example["rejected"])["input_ids"]) if example["rejected"] else 0
    return prompt_len <= MAX_TOKENS and chosen_len <= MAX_TOKENS and rejected_len <= MAX_TOKENS

dataset = dataset.filter(is_valid)


train_dataset = dataset.select(range(2500))
eval_dataset  = dataset.select(range(2500, 3000))

print("Train size:", len(train_dataset))
print("Eval size:", len(eval_dataset))
print("Max prompt tokens:", max(len(tokenizer(x["prompt"])["input_ids"]) for x in train_dataset))
print("Max chosen tokens:", max(len(tokenizer(x["chosen"])["input_ids"]) for x in train_dataset))
print("Max rejected tokens:", max(len(tokenizer(x["rejected"])["input_ids"]) for x in train_dataset))

In [ ]:
from datasets import load_from_disk

TRAIN_PATH = "/content/drive/MyDrive/datasets/dolci_train"
EVAL_PATH  = "/content/drive/MyDrive/datasets/dolci_eval"


train_dataset.save_to_disk(TRAIN_PATH)
eval_dataset.save_to_disk(EVAL_PATH)

print(f"Train dataset saved to {TRAIN_PATH}")
print(f"Eval dataset saved to {EVAL_PATH}")



In [ ]:
from datasets import load_from_disk

# Load datasets
TRAIN_PATH = "/content/drive/MyDrive/datasets/dolci_train"
EVAL_PATH  = "/content/drive/MyDrive/datasets/dolci_eval"

train_dataset = load_from_disk(TRAIN_PATH)
eval_dataset  = load_from_disk(EVAL_PATH)

Fine-tuning

In [ ]:
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cuda"
)


lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
def print_vram_usage():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    max_allocated = torch.cuda.max_memory_allocated() / 1024**3

    print(f"Allocated: {allocated:.2f} GB")
    print(f"Reserved: {reserved:.2f} GB")
    print(f"Max allocated: {max_allocated:.2f} GB")

In [ ]:
!pip uninstall -y wandb
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
from trl import ORPOTrainer, ORPOConfig

training_args = ORPOConfig(
    output_dir="./orpo-llama-1b",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    logging_steps=20,
    save_strategy="no",
    learning_rate=5e-6,
    warmup_ratio=0.05,
    max_length=512,
    max_prompt_length=192,
    fp16=True,
    beta=0.05,
    optim="adamw_torch",
)


trainer = ORPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
)


torch.cuda.reset_peak_memory_stats()


trainer.train()

print_vram_usage()

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Modello salvato in {output_dir}")

Eval

In [ ]:
# RUN INSTEAD OF THE FINE-TUNING BLOCKS

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load the fine-tuned tokenizer
tokenizer = AutoTokenizer.from_pretrained(output_dir)

# Load the fine-tuned model
model = AutoModelForCausalLM.from_pretrained(
    output_dir,
    torch_dtype=torch.float16,
    device_map="auto"
)

print(f"Fine-tuned model and tokenizer loaded successfully from {output_dir}")

In [ ]:
import torch
import torch.nn.functional as F

def compute_logprob_batch(
    model,
    tokenizer,
    prompts,
    responses,
    max_length=1024,
    average_log_prob=True,   # se False usa sum invece di mean
):
    """
    Computes log p(response | prompt) for a batch.

    Returns:
        List[float]: log-probability (mean or sum) for each (prompt, response)
    """

    device = model.device
    pad_token_id = tokenizer.pad_token_id

    batch_input_ids = []
    batch_attention_mask = []
    batch_prompt_lengths = []
    batch_response_lengths = []


    # Tokenization
    for prompt, response in zip(prompts, responses):

        prompt_ids = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            add_special_tokens=False,
        )["input_ids"].squeeze(0)

        response_ids = tokenizer(
            response,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            add_special_tokens=False,
        )["input_ids"].squeeze(0)

        combined_ids = torch.cat([prompt_ids, response_ids], dim=0)

        batch_input_ids.append(combined_ids)
        batch_attention_mask.append(torch.ones_like(combined_ids))

        batch_prompt_lengths.append(prompt_ids.shape[0])
        batch_response_lengths.append(response_ids.shape[0])


    # Padding
    padded_input_ids = torch.nn.utils.rnn.pad_sequence(
        batch_input_ids,
        batch_first=True,
        padding_value=pad_token_id,
    ).to(device)

    padded_attention_mask = torch.nn.utils.rnn.pad_sequence(
        batch_attention_mask,
        batch_first=True,
        padding_value=0,
    ).to(device)


    # Forward pass
    model.eval()
    with torch.no_grad():

        outputs = model(
            input_ids=padded_input_ids,
            attention_mask=padded_attention_mask,
        )

        logits = outputs.logits  # (B, T, V)

        # Causal shift
        shift_logits = logits[:, :-1, :]
        shift_labels = padded_input_ids[:, 1:]

        log_probs = F.log_softmax(shift_logits, dim=-1)

        # Gather log-probs of target tokens
        token_log_probs = torch.gather(
            log_probs,
            dim=-1,
            index=shift_labels.unsqueeze(-1),
        ).squeeze(-1)

        # Mask padding tokens
        loss_mask = shift_labels != pad_token_id
        token_log_probs = token_log_probs * loss_mask

        results = []


        # Extract response scores
        for i in range(len(prompts)):

            prompt_len = batch_prompt_lengths[i]
            response_len = batch_response_lengths[i]

            if response_len == 0:
                results.append(0.0)
                continue

            # Start index of response log-probs
            start = prompt_len - 1
            end = start + response_len

            response_token_log_probs = token_log_probs[i, start:end]

            total_logprob = response_token_log_probs.sum()

            if average_log_prob:
                score = total_logprob / response_len
            else:
                score = total_logprob

            results.append(score.item())

    return results

Compute Pairwise accuracy

In [ ]:
from tqdm import tqdm
import torch

def pairwise_accuracy_batched(
    policy_model,
    tokenizer,
    dataset,
    batch_size=4,
    max_length=1024,
    average_log_prob=True,
):

    correct = 0
    total = len(dataset)

    policy_model.eval()


    with torch.no_grad():

        for i in tqdm(range(0, total, batch_size), desc="Elaborazione batch"):

            batch = dataset[i:i + batch_size]

            prompts = batch["prompt"]

            chosen_responses = batch["chosen"]

            rejected_responses = batch["rejected"]



            # Policy log-probs
            policy_chosen = compute_logprob_batch(
                policy_model,
                tokenizer,
                prompts,
                chosen_responses,
                max_length=max_length,
                average_log_prob=average_log_prob,
            )

            policy_rejected = compute_logprob_batch(
                policy_model,
                tokenizer,
                prompts,
                rejected_responses,
                max_length=max_length,
                average_log_prob=average_log_prob,
            )




            for j in range(len(prompts)):

                chosen_score = policy_chosen[j]
                rejected_score = policy_rejected[j]

                if chosen_score > rejected_score:
                    correct += 1

    return correct / total

acc = pairwise_accuracy_batched(
    policy_model=model,
    tokenizer=tokenizer,
    dataset=eval_dataset,
    batch_size=4,
)

print("\nPairwise accuracy (ORPO):", acc)